[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/ia_p26/blob/main/clase/07_optimization/notebooks/02_algoritmos_y_codigo.ipynb)

# Notebook 2: Algoritmos y Código

En este notebook:
1. Implementamos descenso de gradiente desde cero
2. Exploramos el efecto del learning rate
3. Resolvemos problemas con `scipy.optimize`
4. Optimización con restricciones (Lagrange + scipy)
5. Programación lineal

In [ ]:
# Instalar dependencias (ejecutar en Colab)
# !pip install numpy matplotlib scipy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize, minimize_scalar, linprog

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

COLORS = {"blue": "#2E86AB", "red": "#E94F37", "green": "#27AE60",
          "orange": "#F39C12", "purple": "#8E44AD", "gray": "#7F8C8D"}

---
## 1. Descenso de gradiente desde cero

**Teoría:** [7.3 Descenso de gradiente](https://www.sonder.art/ia_p26/07_optimization/03_algoritmos/#descenso-de-gradiente-sin-restricciones)

La regla es simple:

$$x_{k+1} = x_k - \alpha \nabla f(x_k)$$

- $\nabla f$: dirección de mayor crecimiento
- $-\nabla f$: dirección "cuesta abajo"
- $\alpha$: tamaño del paso (learning rate)

In [ ]:
def gradient_descent(f, grad_f, x0, lr=0.01, n_steps=100):
    """Descenso de gradiente básico.

    Args:
        f: función objetivo f(x) -> float
        grad_f: gradiente grad_f(x) -> array
        x0: punto inicial (array)
        lr: learning rate
        n_steps: número de iteraciones

    Returns:
        trajectory: array (n_steps+1, n) con todos los puntos visitados
    """
    x = x0.copy().astype(float)
    trajectory = [x.copy()]
    for _ in range(n_steps):
        x = x - lr * grad_f(x)
        trajectory.append(x.copy())
    return np.array(trajectory)

In [ ]:
# Probemos con una cuadrática simple: f(x,y) = 3x^2 + y^2
f_quad = lambda x: 3 * x[0]**2 + x[1]**2
grad_quad = lambda x: np.array([6 * x[0], 2 * x[1]])

traj = gradient_descent(f_quad, grad_quad, np.array([3.0, 3.0]), lr=0.08, n_steps=25)

print(f"Inicio:  ({traj[0, 0]:.3f}, {traj[0, 1]:.3f}), f = {f_quad(traj[0]):.3f}")
print(f"Final:   ({traj[-1, 0]:.3f}, {traj[-1, 1]:.3f}), f = {f_quad(traj[-1]):.3f}")
print(f"Óptimo:  (0, 0), f = 0")

---
## 2. Visualizar la trayectoria de GD

In [ ]:
def plot_gd_contour(f_2d, trajectory, x_range=(-4, 4), y_range=(-4, 4),
                    title="Descenso de gradiente", n_levels=20):
    """Grafica contornos de f con la trayectoria de GD."""
    xg = np.linspace(*x_range, 300)
    yg = np.linspace(*y_range, 300)
    X, Y = np.meshgrid(xg, yg)
    Z = f_2d(X, Y)

    fig, ax = plt.subplots(figsize=(8, 7))
    cs = ax.contour(X, Y, Z, levels=n_levels, cmap="viridis", alpha=0.7)
    ax.clabel(cs, inline=True, fontsize=8)

    ax.plot(trajectory[:, 0], trajectory[:, 1], "o-", color=COLORS["red"],
            markersize=4, linewidth=1.5, label="Trayectoria GD")
    ax.scatter(*trajectory[0], color=COLORS["orange"], s=100, zorder=5,
              edgecolors="black", label="Inicio")
    ax.scatter(*trajectory[-1], color=COLORS["green"], s=100, zorder=5,
              edgecolors="black", label="Final")

    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.legend()
    plt.show()

plot_gd_contour(
    lambda X, Y: 3*X**2 + Y**2,
    traj,
    title=r"GD en $f(x,y) = 3x^2 + y^2$, lr=0.08"
)

---
## 3. Ejercicio: Explorar learning rates

**Teoría:** [7.3 El learning rate importa mucho](https://www.sonder.art/ia_p26/07_optimization/03_algoritmos/#el-learning-rate-importa-mucho)

Prueba 3 learning rates diferentes y observa qué pasa.

In [ ]:
learning_rates = [0.01, 0.08, 0.30]  # <-- CHANGE THIS: prueba valores como 0.001, 0.5, etc.

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

xg = np.linspace(-4, 4, 300)
yg = np.linspace(-4, 4, 300)
X, Y = np.meshgrid(xg, yg)
Z = 3*X**2 + Y**2

for ax, lr in zip(axes, learning_rates):
    traj_lr = gradient_descent(f_quad, grad_quad, np.array([3.0, 3.0]), lr=lr, n_steps=30)

    ax.contour(X, Y, Z, levels=20, cmap="viridis", alpha=0.5)
    ax.plot(traj_lr[:, 0], traj_lr[:, 1], "o-", color=COLORS["red"],
            markersize=3, linewidth=1)
    ax.scatter(*traj_lr[0], color=COLORS["orange"], s=80, zorder=5, edgecolors="black")
    ax.scatter(*traj_lr[-1], color=COLORS["green"], s=80, zorder=5, edgecolors="black")
    ax.set_title(f"lr = {lr}")
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)

fig.suptitle("Efecto del learning rate", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

### Reflexión: Learning rate

- El learning rate **crítico** para $f(x,y) = 3x^2 + y^2$ es $\alpha_{\text{crit}} = 2/L_{\max}$ donde $L_{\max} = 6$ (eigenvalor más grande del Hessiano). Así que $\alpha_{\text{crit}} = 1/3 \approx 0.333$.
- Prueba `lr=0.33` — converge lento pero estable. Ahora prueba `lr=0.34` — ¿qué pasa?
- Este es un resultado teórico: GD converge si y solo si $\alpha < 2/L_{\max}$ para cuadráticas.

---
## SGD vs Batch GD

**Teoría:** [7.3 Descenso de gradiente estocástico](https://www.sonder.art/ia_p26/07_optimization/03_algoritmos/#descenso-de-gradiente-estocástico-sgd)

En ML, la función objetivo es una suma sobre datos: $f(w) = \frac{1}{N}\sum_i f_i(w)$.
SGD estima el gradiente con un **mini-batch** aleatorio en lugar de usar todos los datos.
Veamos la diferencia en un problema de regresión lineal sintética.

In [ ]:
# Datos sintéticos: y = 2*x + 1 + ruido
np.random.seed(42)
N = 200
X_data = np.random.randn(N, 1)
y_data = 2 * X_data[:, 0] + 1 + 0.5 * np.random.randn(N)

# Loss: MSE = (1/N) * sum((y - Xw)^2), w = [slope, intercept]
X_aug = np.hstack([X_data, np.ones((N, 1))])  # add bias column

def mse_loss(w):
    return np.mean((y_data - X_aug @ w)**2)

def mse_grad_full(w):
    return -2/N * X_aug.T @ (y_data - X_aug @ w)

def mse_grad_sgd(w, batch_size=32):
    idx = np.random.choice(N, batch_size, replace=False)
    X_b, y_b = X_aug[idx], y_data[idx]
    return -2/len(idx) * X_b.T @ (y_b - X_b @ w)

# Run both
w0 = np.array([0.0, 0.0])
n_steps = 150
lr = 0.05

# Batch GD
losses_gd = []
w_gd = w0.copy()
for _ in range(n_steps):
    losses_gd.append(mse_loss(w_gd))
    w_gd = w_gd - lr * mse_grad_full(w_gd)

# SGD (batch_size=32)
losses_sgd = []
w_sgd = w0.copy()
for _ in range(n_steps):
    losses_sgd.append(mse_loss(w_sgd))
    w_sgd = w_sgd - lr * mse_grad_sgd(w_sgd, batch_size=32)

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(losses_gd, linewidth=2, color=COLORS["blue"], label="Batch GD (N=200)")
ax.plot(losses_sgd, linewidth=1.5, color=COLORS["red"], alpha=0.8, label="SGD (batch=32)")
ax.set_xlabel("Iteración")
ax.set_ylabel("MSE Loss")
ax.set_title("SGD vs Batch GD en regresión lineal")
ax.legend()
ax.set_yscale("log")
plt.show()

print(f"Batch GD final: w = [{w_gd[0]:.3f}, {w_gd[1]:.3f}]")
print(f"SGD final:      w = [{w_sgd[0]:.3f}, {w_sgd[1]:.3f}]")
print(f"Verdadero:      w = [2.000, 1.000]")

---
## 4. Rosenbrock: cuando GD sufre

**Teoría:** [7.3 Métodos de segundo orden](https://www.sonder.art/ia_p26/07_optimization/03_algoritmos/#métodos-de-segundo-orden--intuición) | [7.4 Patrón 2](https://www.sonder.art/ia_p26/07_optimization/04_ejemplos_python/#patrón-2-minimización-multidimensional--minimize)

La función de Rosenbrock es famosa por su "valle banana" donde GD avanza muy lento:

$$f(x,y) = (1-x)^2 + 100(y - x^2)^2$$

Mínimo global en $(1, 1)$ con $f = 0$.

In [ ]:
rosenbrock = lambda x: (1 - x[0])**2 + 100 * (x[1] - x[0]**2)**2
grad_rosen = lambda x: np.array([
    -2 * (1 - x[0]) - 400 * x[0] * (x[1] - x[0]**2),
    200 * (x[1] - x[0]**2),
])

traj_rosen = gradient_descent(rosenbrock, grad_rosen,
                               np.array([-1.5, 2.0]), lr=0.001, n_steps=5000)

print(f"Después de 5000 pasos: ({traj_rosen[-1, 0]:.4f}, {traj_rosen[-1, 1]:.4f})")
print(f"Distancia al óptimo (1,1): {np.linalg.norm(traj_rosen[-1] - [1, 1]):.4f}")

In [ ]:
# Visualizar
rosen_2d = lambda X, Y: (1 - X)**2 + 100 * (Y - X**2)**2

xg = np.linspace(-2, 2, 400)
yg = np.linspace(-1, 3, 400)
X, Y = np.meshgrid(xg, yg)
Z = rosen_2d(X, Y)

fig, ax = plt.subplots(figsize=(9, 7))
cs = ax.contour(X, Y, np.log1p(Z), levels=30, cmap="inferno", alpha=0.7)

# Subsample trajectory
step = max(1, len(traj_rosen) // 200)
traj_sub = traj_rosen[::step]
ax.plot(traj_sub[:, 0], traj_sub[:, 1], "-", color=COLORS["blue"],
        linewidth=1, alpha=0.7)
ax.scatter(*traj_rosen[0], color=COLORS["orange"], s=100, zorder=5, edgecolors="black", label="Inicio")
ax.scatter(*traj_rosen[-1], color=COLORS["green"], s=100, zorder=5, edgecolors="black", label="Final")
ax.scatter(1, 1, color=COLORS["red"], s=120, zorder=5, marker="*", edgecolors="black", label="Óptimo (1,1)")

ax.set_title("GD en Rosenbrock (5000 pasos, lr=0.001)")
ax.legend()
plt.show()

### Reflexión: Rosenbrock y segundo orden

- GD usa 5000 pasos y no llega al óptimo. L-BFGS-B llega en ~50 evaluaciones. ¿Por qué?
- L-BFGS-B usa **curvatura aproximada** (información de segundo orden) para tomar pasos más inteligentes.
- En el "valle banana", la curvatura es muy diferente en la dirección del valle vs perpendicular a él. GD no lo sabe; L-BFGS-B sí.

---
## 5. scipy.optimize: minimización 1D

**Teoría:** [7.4 Patrón 1](https://www.sonder.art/ia_p26/07_optimization/04_ejemplos_python/#patrón-1-minimización-1d--minimize_scalar)

Para problemas 1D, `minimize_scalar` es directo.

In [ ]:
f_1d = lambda x: (x - 3)**2 + 2 * np.sin(5 * x)

result = minimize_scalar(f_1d, bounds=(0, 6), method="bounded")
print(f"Mínimo encontrado: x = {result.x:.4f}")
print(f"f(x) = {result.fun:.4f}")

# Visualizar
x = np.linspace(-1, 7, 800)
fig, ax = plt.subplots()
ax.plot(x, f_1d(x), linewidth=2, color=COLORS["blue"],
        label=r"$f(x) = (x-3)^2 + 2\sin(5x)$")
ax.scatter(result.x, result.fun, color=COLORS["red"], s=120, zorder=5,
           edgecolors="black")
ax.annotate(f"Mínimo: ({result.x:.2f}, {result.fun:.2f})",
            xy=(result.x, result.fun), xytext=(result.x + 1, result.fun + 3),
            arrowprops=dict(arrowstyle="->"), fontsize=11, color=COLORS["red"])
ax.set_title("Minimización 1D con scipy")
ax.legend()
plt.show()

---
## 6. scipy.optimize: minimización 2D (Rosenbrock)

**Teoría:** [7.4 Patrón 2](https://www.sonder.art/ia_p26/07_optimization/04_ejemplos_python/#patrón-2-minimización-multidimensional--minimize)

Comparemos nuestro GD casero con scipy (que usa L-BFGS-B por defecto).

In [ ]:
result_scipy = minimize(rosenbrock, x0=np.array([-1.5, 2.0]), method="L-BFGS-B")

print(f"scipy L-BFGS-B:")
print(f"  Solución: ({result_scipy.x[0]:.6f}, {result_scipy.x[1]:.6f})")
print(f"  f(x) = {result_scipy.fun:.10f}")
print(f"  Evaluaciones de f: {result_scipy.nfev}")
print(f"\nNuestro GD (5000 pasos):")
print(f"  Solución: ({traj_rosen[-1, 0]:.6f}, {traj_rosen[-1, 1]:.6f})")
print(f"  f(x) = {rosenbrock(traj_rosen[-1]):.10f}")

---
## 7. Optimización con restricciones: Lagrange

**Teoría:** [7.3 Multiplicadores de Lagrange](https://www.sonder.art/ia_p26/07_optimization/03_algoritmos/#multiplicadores-de-lagrange-restricciones-de-igualdad)

**Problema:** $\min x^2 + y^2$ sujeto a $x + y = 1$

**Por Lagrange:**

$\mathcal{L}(x,y,\lambda) = x^2 + y^2 + \lambda(x + y - 1)$

Resolviendo:
- $\partial_x: 2x + \lambda = 0$
- $\partial_y: 2y + \lambda = 0$
- $\partial_\lambda: x + y - 1 = 0$

→ $x = y = 1/2$, $f^* = 1/2$

In [ ]:
# Verificar con scipy
f_constrained = lambda x: x[0]**2 + x[1]**2
constraint = {"type": "eq", "fun": lambda x: x[0] + x[1] - 1}

result_lag = minimize(f_constrained, x0=[0.0, 0.0], constraints=constraint)

print(f"Solución analítica: (0.5, 0.5), f = 0.5")
print(f"Solución scipy:     ({result_lag.x[0]:.4f}, {result_lag.x[1]:.4f}), f = {result_lag.fun:.4f}")

In [ ]:
# Visualizar: contornos de f con la restricción
xg = np.linspace(-1, 2, 300)
yg = np.linspace(-1, 2, 300)
X, Y = np.meshgrid(xg, yg)
Z = X**2 + Y**2

fig, ax = plt.subplots(figsize=(7, 7))
cs = ax.contour(X, Y, Z, levels=15, cmap="viridis", alpha=0.7)
ax.clabel(cs, inline=True, fontsize=8)

# Constraint line: x + y = 1
ax.plot(xg, 1 - xg, "--", color=COLORS["red"], linewidth=2, label="x + y = 1")

# Solution
ax.scatter(0.5, 0.5, color=COLORS["red"], s=150, zorder=5, edgecolors="black",
           label="Óptimo (0.5, 0.5)")

ax.set_title(r"$\min\ x^2 + y^2$ sujeto a $x + y = 1$")
ax.legend()
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.show()

---
## 8. scipy con restricciones (dict)

Puedes pasar restricciones de igualdad y desigualdad como diccionarios.

In [ ]:
# Ejemplo: min x^2 + y^2  s.t.  x + y >= 2, x >= 0, y >= 0
f_ex = lambda x: x[0]**2 + x[1]**2

constraints = [
    {"type": "ineq", "fun": lambda x: x[0] + x[1] - 2},  # x+y >= 2  (ineq means >= 0)
]
bounds = [(0, None), (0, None)]

result_ineq = minimize(f_ex, x0=[2.0, 2.0], constraints=constraints, bounds=bounds)
print(f"Solución: ({result_ineq.x[0]:.4f}, {result_ineq.x[1]:.4f})")
print(f"f(x) = {result_ineq.fun:.4f}")
print(f"x+y = {result_ineq.x[0] + result_ineq.x[1]:.4f} (debe ser >= 2)")

---
## 9. Programación lineal con `linprog`

**Teoría:** [7.3 Método simplex](https://www.sonder.art/ia_p26/07_optimization/03_algoritmos/#método-simplex-programación-lineal)

Resolvemos el problema de producción:

$$\max\ 5x_1 + 4x_2 \quad \text{s.t.} \quad 6x_1 + 4x_2 \leq 24,\quad x_1 + 2x_2 \leq 6,\quad x_1, x_2 \geq 0$$

`linprog` **minimiza**, así que negamos $c$.

In [ ]:
c = [-5, -4]              # min -5x1 - 4x2 = max 5x1 + 4x2
A_ub = [[6, 4], [1, 2]]
b_ub = [24, 6]
bounds = [(0, None), (0, None)]

result_lp = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method="highs")

print(f"Solución: x1 = {result_lp.x[0]:.2f}, x2 = {result_lp.x[1]:.2f}")
print(f"Ganancia máxima: {-result_lp.fun:.2f}")

In [ ]:
# Visualizar la región factible + solución
fig, ax = plt.subplots(figsize=(8, 7))

x1 = np.linspace(0, 5, 400)
c1 = (24 - 6*x1) / 4
c2 = (6 - x1) / 2

ax.plot(x1, c1, color=COLORS["blue"], linewidth=2, label=r"$6x_1 + 4x_2 \leq 24$")
ax.plot(x1, c2, color=COLORS["green"], linewidth=2, label=r"$x_1 + 2x_2 \leq 6$")

y_upper = np.minimum(c1, c2)
y_upper = np.maximum(y_upper, 0)
ax.fill_between(x1, 0, y_upper, where=(y_upper >= 0) & (x1 >= 0),
                alpha=0.2, color=COLORS["blue"], label="Región factible")

sol = result_lp.x
ax.scatter(sol[0], sol[1], color=COLORS["red"], s=200, zorder=6,
           edgecolors="black", linewidths=2, marker="*")
ax.annotate(f"Óptimo: ({sol[0]:.1f}, {sol[1]:.1f})\nz = {-result_lp.fun:.0f}",
            xy=(sol[0], sol[1]), xytext=(sol[0]+0.3, sol[1]+0.6),
            arrowprops=dict(arrowstyle="->"), fontsize=11, color=COLORS["red"],
            fontweight="bold")

ax.set_xlim(-0.5, 5)
ax.set_ylim(-0.5, 5)
ax.set_title("Programación lineal: solución con scipy.linprog")
ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")
ax.legend()
plt.show()

### Reflexión: Programación lineal

- La solución siempre está en un **vértice** del politopo (región factible).
- Cambia `c = [-4, -5]` (intercambia ganancias). ¿La solución salta a otro vértice?
- Cambia `b_ub = [24, 10]` (más recurso B). ¿La región factible crece? ¿Cambia el óptimo?

---
## Diferenciación automática (autodiff)

**Teoría:** [7.4 Autodiff](https://www.sonder.art/ia_p26/07_optimization/04_ejemplos_python/#diferenciación-automática-autodiff)

Hasta ahora calculamos gradientes **a mano**. Autodiff los calcula exactamente y automáticamente — es lo que hace `loss.backward()` en PyTorch.

In [ ]:
# Autodiff con PyTorch — compara con gradiente analítico de Rosenbrock
try:
    import torch

    # Mismo punto que usamos antes
    xy = torch.tensor([-1.5, 2.0], requires_grad=True)
    loss = (1 - xy[0])**2 + 100 * (xy[1] - xy[0]**2)**2
    loss.backward()

    grad_autodiff = xy.grad.numpy()
    grad_manual = grad_rosen(np.array([-1.5, 2.0]))

    print("Gradiente de Rosenbrock en (-1.5, 2.0):")
    print(f"  Autodiff (PyTorch): [{grad_autodiff[0]:.4f}, {grad_autodiff[1]:.4f}]")
    print(f"  Analítico (a mano): [{grad_manual[0]:.4f}, {grad_manual[1]:.4f}]")
    print(f"  ¿Coinciden? {np.allclose(grad_autodiff, grad_manual)}")
    print("\nEsto es lo que hace loss.backward() en cada paso de entrenamiento.")
except ImportError:
    print("PyTorch no está instalado.")
    print("En Colab: !pip install torch")
    print("El gradiente analítico de Rosenbrock en (-1.5, 2.0) es:")
    grad_manual = grad_rosen(np.array([-1.5, 2.0]))
    print(f"  [{grad_manual[0]:.4f}, {grad_manual[1]:.4f}]")

---
## Optimización entera (opcional)

**Teoría:** [7.2 Optimización entera / discreta](https://www.sonder.art/ia_p26/07_optimization/02_paisaje_y_conceptos/#optimización-entera--discreta)

Cuando las variables deben ser **enteras**, no puedes simplemente redondear la solución continua.
`scipy.optimize.milp` resuelve programas lineales con variables enteras (MIP).

In [ ]:
# Mini knapsack: max 3x1 + 4x2  s.t.  2x1 + 3x2 <= 7,  x1,x2 enteros >= 0
from scipy.optimize import milp, LinearConstraint, Bounds

c = [-3, -4]  # milp minimiza, así que negamos
constraints = LinearConstraint([[2, 3]], ub=7)
integrality = [1, 1]  # 1 = variable entera
bounds = Bounds(lb=0)

result_mip = milp(c, constraints=constraints, integrality=integrality, bounds=bounds)
print(f"Solución entera: x1={result_mip.x[0]:.0f}, x2={result_mip.x[1]:.0f}")
print(f"Ganancia máxima: {-result_mip.fun:.0f}")

# Compara con la relajación continua (sin integrality)
result_relax = milp(c, constraints=constraints, bounds=bounds)
print(f"\nRelajación continua: x1={result_relax.x[0]:.2f}, x2={result_relax.x[1]:.2f}")
print(f"Ganancia (continua): {-result_relax.fun:.2f}")
print("Redondear la continua NO siempre da la solución entera óptima.")

---
## 10. Ejercicio capstone

**Teoría:** [7.4 Ejercicio capstone](https://www.sonder.art/ia_p26/07_optimization/04_ejemplos_python/#ejercicio-capstone-a-mano-y-con-scipy)

**Problema:** Una empresa quiere minimizar costos de transporte:

$$f(x_1, x_2) = 2x_1^2 + 3x_2^2 + x_1 x_2$$

sujeto a $x_1 + x_2 \geq 10$, $x_1, x_2 \geq 0$.

**Tareas:**
1. Resuelve con Lagrange (asumiendo restricción activa)
2. Verifica con scipy
3. Visualiza la solución en contornos

In [ ]:
# 1. Tu solución analítica (escribe los valores)
x1_analitico = 6.25   # <-- CHANGE THIS
x2_analitico = 3.75   # <-- CHANGE THIS

f_capstone = lambda x: 2*x[0]**2 + 3*x[1]**2 + x[0]*x[1]
print(f"Solución analítica: ({x1_analitico}, {x2_analitico})")
print(f"f = {f_capstone([x1_analitico, x2_analitico]):.4f}")

In [ ]:
# 2. Verificar con scipy
constraint_cap = {"type": "ineq", "fun": lambda x: x[0] + x[1] - 10}
bounds_cap = [(0, None), (0, None)]

result_cap = minimize(f_capstone, x0=[5.0, 5.0], constraints=constraint_cap, bounds=bounds_cap)

print(f"scipy: ({result_cap.x[0]:.4f}, {result_cap.x[1]:.4f})")
print(f"f = {result_cap.fun:.4f}")
print(f"x1 + x2 = {result_cap.x[0] + result_cap.x[1]:.4f}")

In [ ]:
# 3. Visualizar
xg = np.linspace(0, 12, 300)
yg = np.linspace(0, 12, 300)
X, Y = np.meshgrid(xg, yg)
Z = 2*X**2 + 3*Y**2 + X*Y

fig, ax = plt.subplots(figsize=(8, 7))
cs = ax.contour(X, Y, Z, levels=25, cmap="viridis", alpha=0.7)
ax.clabel(cs, inline=True, fontsize=8)

# Constraint: x + y = 10
ax.plot(xg, 10 - xg, "--", color=COLORS["red"], linewidth=2, label="x + y = 10")
ax.fill_between(xg, 10 - xg, 12, alpha=0.1, color=COLORS["green"], label="Factible")

# Solution
sol_cap = result_cap.x
ax.scatter(sol_cap[0], sol_cap[1], color=COLORS["red"], s=150, zorder=5,
           edgecolors="black", marker="*", label=f"Óptimo ({sol_cap[0]:.2f}, {sol_cap[1]:.2f})")

ax.set_xlim(0, 12)
ax.set_ylim(0, 12)
ax.set_title(r"$\min\ 2x_1^2 + 3x_2^2 + x_1 x_2$ s.t. $x_1 + x_2 \geq 10$")
ax.legend()
ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")
plt.show()